<a href="https://colab.research.google.com/github/SamanTarique/flyrank-01-ml-2026/blob/main/work/notebooks/w06_validation_audit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation Audit (Week 6)

This notebook audits my own Week-5 model the way we audited FlyRank's research paper: same
questions, same standard of evidence, no exceptions for being my own work.

**Note on the Week-4 CSV:** Week-5 loaded `work/outputs/baseline_action_score.csv` (a committed
output file) to compare the model against the baseline. That comparison is *not* repeated here.
Everything below is computed straight from the raw warehouse tables so the whole audit stands on
one source of truth — no derived/output CSVs, no re-scaling drift between weeks.


## 1. Two paper findings + my methodology questions

*Pick two findings from FlyRank's research paper. For each, in your own words, write the
methodology question you'd ask a colleague about it — where does the label come from? does the
validation design support the claim being made? Keep it respectful and concrete: the same
question you'd want asked of your own Week-5 notebook below.*

Fill this in from your notes on the live session — I don't have the paper's text in front of me,
so I can't summarize its findings for you accurately. Two prompts to structure each entry:

---

**Finding 1:** [**Insert your first finding from the FlyRank paper here, in one sentence.**]

**My methodology question 1:** [**Insert your methodology question for Finding 1 here (e.g., label source, split design, sample size, confound).**]

---

**Finding 2:** [**Insert your second finding from the FlyRank paper here, in one sentence.**]

**My methodology question 2:** [**Insert your methodology question for Finding 2 here (e.g., label source, split design, sample size, confound).**]

---


## 2. My model under an honest split — before / after

**Before:** Week-5 used a stratified *random* split. That's safe against class-imbalance but not
against **client leakage** — `client_hash_id` exists in the raw tables (it was excluded as a
*feature* in the Week-4 data contract, but it was never used as a *grouping key* for the split).
If one client's content items land in both train and test, the model can partly memorize that
client's baseline position/behavior instead of learning the general pattern.

**After:** Same features, same target, same model family and hyperparameters as Week-5 — the
only thing that changes is the split: `GroupShuffleSplit` grouped on `client_hash_id`, so every
client's rows sit entirely in train or entirely in test. If ROC-AUC drops going from "before" to
"after", that drop is the leakage the random split was hiding.

Run this in Colab with your own `hf` token — it rebuilds `model_df` from the raw warehouse
tables, the same way Week-5 did.


In [ ]:
import os, getpass
import numpy as np
import pandas as pd
import duckdb

from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
TEST_SIZE = 0.2

HF_token = os.environ.get('saman_tech')
if not HF_token:
    try:
        from google.colab import userdata
        HF_token = userdata.get('saman_tech')
    except Exception:
        pass
HF_token = HF_token or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_token}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'


In [ ]:


model_df = con.sql(f"""
    SELECT
        dc.content_hash_id,
        dc.client_hash_id,
        dc.content_type,
        dc.char_count,
        dc.last_optimized_date,
        MAX(fcq.content_total_impressions_90d) AS content_total_impressions_90d,
        MAX(fcq.content_visible_query_count)   AS content_visible_query_count,
        MAX(fcq.rare_query_count)              AS rare_query_count,
        AVG(fcq.avg_position_prev30)           AS avg_position_prev30,
        AVG(fcq.avg_position_last30)           AS avg_position_last30
    FROM '{REL}/dim_content.parquet' dc
    JOIN '{REL}/fact_content_query_90d.parquet' fcq
      ON dc.content_hash_id = fcq.content_hash_id
    WHERE dc.is_published IS TRUE AND dc.is_deleted IS NOT TRUE
    GROUP BY dc.content_hash_id, dc.client_hash_id, dc.content_type,
             dc.char_count, dc.last_optimized_date
""").df()

model_df['declined'] = (model_df['avg_position_last30'] > model_df['avg_position_prev30']).astype(int)
model_df['staleness_days'] = (
    pd.Timestamp.today().normalize() - pd.to_datetime(model_df['last_optimized_date'])
).dt.days

TARGET = 'declined'
FEATURES_NUM = ['staleness_days', 'char_count', 'content_total_impressions_90d',
                'content_visible_query_count', 'rare_query_count']
FEATURES_CAT = ['content_type']
GROUP = 'client_hash_id'

print('Rows:', len(model_df), ' Unique clients:', model_df[GROUP].nunique())
print('Class balance:\n', model_df[TARGET].value_counts(normalize=True))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 133801  Unique clients: 51
Class balance:
 declined
0    0.527866
1    0.472134
Name: proportion, dtype: float64


In [ ]:
preprocess = ColumnTransformer([
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore'), FEATURES_CAT),
])

def make_pipe():
    return Pipeline([
        ('prep', preprocess),
        ('clf', RandomForestClassifier(random_state=RANDOM_STATE)),
    ])

def fit_eval(train_df, test_df, label):
    X_train, y_train = train_df[FEATURES_NUM + FEATURES_CAT], train_df[TARGET]
    X_test,  y_test  = test_df[FEATURES_NUM + FEATURES_CAT],  test_df[TARGET]
    pipe = make_pipe()
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred  = pipe.predict(X_test)
    auc = roc_auc_score(y_test, proba)
    cm = confusion_matrix(y_test, pred)
    print(f'--- {label} ---')
    print('Train rows:', len(train_df), ' Test rows:', len(test_df))
    print('Test ROC-AUC:', round(auc, 3))
    display(pd.DataFrame(cm, index=['actual_stable', 'actual_declined'],
                          columns=['pred_stable', 'pred_declined']))
    return pipe, X_test, y_test, auc

# BEFORE — Week-5's split: stratified random, no grouping
before_train, before_test = train_test_split(
    model_df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=model_df[TARGET]
)
before_pipe, before_Xtest, before_ytest, before_auc = fit_eval(
    before_train, before_test, 'BEFORE — stratified random split (Week-5)'
)


--- BEFORE — stratified random split (Week-5) ---
Train rows: 107040  Test rows: 26761
Test ROC-AUC: 0.64


,pred_stable,pred_declined
actual_stable,8910,5216
actual_declined,5430,7205


In [ ]:
# AFTER — honest split: grouped by client_hash_id, no client appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(model_df, groups=model_df[GROUP]))
after_train, after_test = model_df.iloc[train_idx], model_df.iloc[test_idx]

overlap = set(after_train[GROUP]) & set(after_test[GROUP])
print('Clients present in both train and test after grouping:', len(overlap), '(should be 0)')

after_pipe, after_Xtest, after_ytest, after_auc = fit_eval(
    after_train, after_test, 'AFTER — client-grouped split (honest)'
)

print(f"\nROC-AUC before (random split): {before_auc:.3f}")
print(f"ROC-AUC after  (grouped split): {after_auc:.3f}")
print(f"Change: {after_auc - before_auc:+.3f}")


Clients present in both train and test after grouping: 0 (should be 0)
--- AFTER — client-grouped split (honest) ---
Train rows: 125071  Test rows: 8730
Test ROC-AUC: 0.654


,pred_stable,pred_declined
actual_stable,3426,1518
actual_declined,1864,1922



ROC-AUC before (random split): 0.640
ROC-AUC after  (grouped split): 0.654
Change: +0.013


**Reading the before/after:** run the cells above and record the actual gap. If AUC drops from
before → after, that's the honest number — the random split's higher score was partly the model
recognizing a client it had already seen, not the pattern I actually care about. Write the
observed numbers here once you've run it, in careful language (e.g. *"under a client-grouped
split, ROC-AUC was observed to drop from X to Y, consistent with client-level leakage in the
random split"*) rather than restating the Week-5 claim unchanged.


## 3. Leakage audit

*Go feature by feature. For each, ask: could this feature "know" about the label window? Is it
computed over a period that overlaps `avg_position_last30`?*


In [ ]:
feature_windows = {
    'staleness_days': 'as-of last_optimized_date vs today — fixed at scoring time, no overlap with the 30d label windows.',
    'char_count': 'static content attribute — no time window, no leakage.',
    'content_type': 'static content attribute — no time window, no leakage.',
    'content_total_impressions_90d': 'ROLLING 90-DAY WINDOW — likely overlaps the last-30-day window used for the label. Flag below.',
    'content_visible_query_count': 'depends on the same 90d fact table — same overlap risk as above.',
    'rare_query_count': 'depends on the same 90d fact table — same overlap risk as above.',
}
for feat, note in feature_windows.items():
    print(f'{feat}: {note}')


staleness_days: as-of last_optimized_date vs today — fixed at scoring time, no overlap with the 30d label windows.
char_count: static content attribute — no time window, no leakage.
content_type: static content attribute — no time window, no leakage.
content_total_impressions_90d: ROLLING 90-DAY WINDOW — likely overlaps the last-30-day window used for the label. Flag below.
content_visible_query_count: depends on the same 90d fact table — same overlap risk as above.
rare_query_count: depends on the same 90d fact table — same overlap risk as above.


**Finding:** three of the five features (`content_total_impressions_90d`,
`content_visible_query_count`, `rare_query_count`) are aggregated over a 90-day window that
almost certainly overlaps the 30-day window used to compute `avg_position_last30` — the same days
that define the label are inside the days that define the feature. That's not the same failure as
client leakage (it doesn't make the model memorize an *identity*), but it does let the feature
partly encode the outcome it's supposed to be predicting, which inflates the score in a way a
production system — scoring content *before* that 90-day window closes — would never get to use.

**What I'd do next (not done here, flagged for the record):** re-derive these three features from
only the 90-day window that ends *before* `avg_position_prev30` starts, so every feature is
strictly pre-label in time, then re-run the before/after comparison above on that corrected
feature set.


## 4. Claim rewrite

*Take claims from my earlier weeks that go further than the evidence supports, and rewrite them
in safe language: observed, measured, directional, decision-support — never causal, never
"predicting Google."*


### Original (Week-5):
```
"CV picked Random Forest (mean CV ROC-AUC 0.607). On test it scored 0.598 vs the baseline's 0.446 — a clear win."
```

### Rewritten:
```
Under a stratified random split, the Random Forest model's *observed* ROC-AUC on the held-out test rows (0.598) was directional and higher than the heuristic baseline (0.446) on the same rows. That comparison used a random split that does not control for client-level leakage — see Section 2 — so the size of the gap should be read as an upper bound, not a production estimate.
```

---

### Original (Week-2):
```
"Machine learning can learn these patterns from historical data and provide more accurate prioritization than a fixed rule or a single if-statement."
```

### Rewritten:
```
Based on the observed feature-importance results, the model's ranking leaned directionally on staleness far more than the heuristic's fixed 50/50 volume-staleness weighting. This is a decision-support signal for how the heuristic's weights might be revisited — it is not evidence that the model's overall priority order is more accurate than the heuristic's in production, since neither has been validated against a real review outcome.
```

## Self-check

Before you submit, confirm each line honestly:

- [ ] Named two paper findings and a concrete, respectful methodology question for each
- [ ] Re-ran my Week-5 model under a client-grouped (honest) split, with an actual before/after
      ROC-AUC comparison — no output CSV from a prior week loaded anywhere in this notebook
- [ ] Leakage audit covers every feature, not just the obvious ones
- [ ] Every rewritten claim uses observed / measured / directional / decision-support language
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Committed to my repo under `work/notebooks/w06_validation_audit.ipynb` — then submit the
      repo URL on the card. Done.
